# 🤖 Smart Trader – Agente Autónomo de Trading com Q-Learning
**Agentes Autónomos – Projeto Final 2025/2026**

---

## Descrição do Projeto

Este notebook implementa um **agente autónomo de trading** que aprende a comprar e vender 
recursos num mercado simulado, utilizando **Q-Learning** (aprendizagem por reforço).

O projeto explora **três personalidades** de agente com funções de recompensa distintas:

| Personalidade | Estratégia |
|:---|:---|
| 🔵 **Seguro** | Averso ao risco – penaliza perdas com intensidade dupla |
| 🔴 **Arrojado** | Tolerante ao risco – recompensa lucros altos, suaviza perdas |
| 🟢 **Day Trader** | Impaciente – penaliza posições abertas por muito tempo |

---

## Estrutura do Projeto

```
smart_trader/
├── environment.py      # Ambiente de mercado (MarketEnvironment)
├── agent.py            # Agente Q-Learning + 3 personalidades
├── visualizations.py   # Funções de plotagem e análise
└── smart_trader.ipynb  # Este notebook (ponto de entrada)
```


## 1. Importações e Configuração

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

# Adicionar o diretório ao path para importar os módulos do projeto
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from environment import MarketEnvironment, generate_price_series
from agent import SafeAgent, BoldAgent, DayTraderAgent
from visualizations import (
    plot_prices, plot_learning_curves,
    plot_portfolio_comparison, plot_agent_decisions,
    print_summary_table
)

# Configuração de estilo dos gráficos
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#FAFAFA',
    'font.family':      'sans-serif',
    'axes.titlesize':   13,
})

print('✅ Módulos importados com sucesso.')
print(f'   NumPy version: {np.__version__}')

## 2. Geração do Ambiente de Mercado

In [ ]:
# ──────────────────────────────────────────────────────────────
# Geração da série de preços sintética
# A série combina tendências cíclicas + ruído gaussiano,
# simulando um ativo com volatilidade realista.
# ──────────────────────────────────────────────────────────────

N_STEPS = 500   # dias de mercado
SEED    = 42    # reprodutibilidade
WINDOW  = 10    # janela da média móvel

prices = generate_price_series(n_steps=N_STEPS, seed=SEED)

print(f'Série gerada: {len(prices)} dias')
print(f'Preço mínimo : {prices.min():.2f} €')
print(f'Preço máximo : {prices.max():.2f} €')
print(f'Preço médio  : {prices.mean():.2f} €')

plot_prices(prices, window=WINDOW)

## 3. Espaço de Estados e Ações

O ambiente é modelado como um **Processo de Decisão de Markov (MDP)**:

### Estados (18 estados discretos)
O estado observável é um tuplo de 3 componentes:

| Componente | Valores | Significado |
|:---|:---|:---|
| Variação diária | {-1, 0, +1} | Preço desceu / estável / subiu |
| Distância à MA-10 | {-1, 0, +1} | Preço abaixo / próximo / acima da média móvel |
| Posição em carteira | {0, 1} | Sem ação / com ação comprada |

### Ações (3 ações)
| Código | Ação | Efeito |
|:---|:---|:---|
| 0 | **HOLD** | Não fazer nada |
| 1 | **BUY** | Comprar 1 unidade (se sem carteira) |
| 2 | **SELL** | Vender 1 unidade (se com carteira) |

### Algoritmo: Q-Learning
A atualização da Q-Table segue a equação de Bellman:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \cdot \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s,a) \right]$$

Onde:
- $\alpha$ = taxa de aprendizagem
- $\gamma$ = fator de desconto  
- $r$ = recompensa imediata (moldada pela personalidade)


## 4. Treino dos Três Agentes

In [ ]:
# ──────────────────────────────────────────────────────────────
# Hiperparâmetros comuns a todos os agentes
# ──────────────────────────────────────────────────────────────

N_EPISODES    = 600    # episódios de treino
ALPHA         = 0.15   # taxa de aprendizagem
GAMMA         = 0.95   # fator de desconto
EPS_START     = 1.0    # exploração inicial (100%)
EPS_END       = 0.05   # exploração mínima (5%)
EPS_DECAY     = (EPS_START - EPS_END) / N_EPISODES

AGENT_KWARGS = dict(
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon_start=EPS_START,
    epsilon_end=EPS_END,
    epsilon_decay=EPS_DECAY,
)

print('Configuração de treino:')
print(f'  Episódios    : {N_EPISODES}')
print(f'  Alpha (α)    : {ALPHA}')
print(f'  Gamma (γ)    : {GAMMA}')
print(f'  ε inicial    : {EPS_START}  →  ε final: {EPS_END}')
print(f'  Decaimento ε : {EPS_DECAY:.5f} por episódio')

In [ ]:
# ──────────────────────────────────────────────────────────────
# Treino do Agente Seguro
# ──────────────────────────────────────────────────────────────

print('\n' + '='*55)
print('  A treinar: Agente SEGURO 🔵')
print('='*55)

env_safe = MarketEnvironment(prices, window=WINDOW)
safe_agent = SafeAgent(env_safe, **AGENT_KWARGS)
safe_agent.train(n_episodes=N_EPISODES, verbose=True)

In [ ]:
# ──────────────────────────────────────────────────────────────
# Treino do Agente Arrojado
# ──────────────────────────────────────────────────────────────

print('\n' + '='*55)
print('  A treinar: Agente ARROJADO 🔴')
print('='*55)

env_bold = MarketEnvironment(prices, window=WINDOW)
bold_agent = BoldAgent(env_bold, **AGENT_KWARGS)
bold_agent.train(n_episodes=N_EPISODES, verbose=True)

In [ ]:
# ──────────────────────────────────────────────────────────────
# Treino do Agente Day Trader
# ──────────────────────────────────────────────────────────────

print('\n' + '='*55)
print('  A treinar: Agente DAY TRADER 🟢')
print('='*55)

env_dt = MarketEnvironment(prices, window=WINDOW)
dt_agent = DayTraderAgent(env_dt, **AGENT_KWARGS)
dt_agent.train(n_episodes=N_EPISODES, verbose=True)

agents = [safe_agent, bold_agent, dt_agent]
print('\n✅ Treino concluído para todos os agentes.')

## 5. Análise das Curvas de Aprendizagem

In [ ]:
# Comparação das curvas de aprendizagem dos 3 agentes
# A suavização (média móvel de 50 ep.) torna a tendência mais clara.

plot_learning_curves(agents)

**Interpretação:**
- O **Agente Seguro** converge para recompensas mais estáveis mas baixas (evita operações arriscadas)
- O **Agente Arrojado** apresenta maior variância (aposta em movimentos maiores)
- O **Day Trader** converge rapidamente mas para valores negativos inicialmente (penaliza o holding)


## 6. Inspeção da Q-Table Aprendida

In [ ]:
# Mostra a Q-Table do agente Seguro como exemplo
# Cada linha é um estado; a ação marcada com ▶ é a política ótima aprendida

safe_agent.print_q_table()

In [ ]:
bold_agent.print_q_table()

In [ ]:
dt_agent.print_q_table()

## 7. Avaliação Final (Política Gananciosa, ε = 0)

In [ ]:
# Após o treino, avaliamos cada agente com ε=0:
# o agente escolhe sempre a ação de maior valor Q (sem exploração aleatória)

results = {}

for agent in agents:
    r = agent.evaluate()
    results[agent.name] = r
    print(f'[{agent.name}]  Retorno: {r["total_return"]:+.2f} €  '
          f'({r["return_pct"]:+.2f}%)  |  Operações: {r["n_trades"]}')

print_summary_table(results)

## 8. Comparação de Portfolios

In [ ]:
plot_portfolio_comparison(prices, results, window=WINDOW)

## 9. Decisões de Cada Agente sobre o Gráfico de Preços

In [ ]:
# Visualizar as decisões BUY/SELL de cada agente sobre a série de preços

for agent in agents:
    plot_agent_decisions(prices, results[agent.name], agent.name, window=WINDOW)

## 10. Análise Comparativa das Personalidades

In [ ]:
# Distribuição de ações por agente
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors_bar = ['#BDBDBD', '#4CAF50', '#F44336']
labels_bar  = ['HOLD', 'BUY', 'SELL']
agent_colors = ['#2196F3', '#F44336', '#4CAF50']

for ax, agent, acolor in zip(axes, agents, agent_colors):
    actions = results[agent.name]['actions_log']
    counts  = [actions.count(a) for a in [0, 1, 2]]
    pcts    = [c / len(actions) * 100 for c in counts]
    bars = ax.bar(labels_bar, pcts, color=colors_bar, edgecolor='white', linewidth=1.5)
    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=11)
    ax.set_title(f'Agente: {agent.name}', fontsize=12, fontweight='bold', color=acolor)
    ax.set_ylabel('% de dias')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Distribuição de Ações por Personalidade', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_action_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('  → Gráfico guardado: output_action_dist.png')

In [ ]:
# Convergência: ε ao longo dos episódios
eps_start, eps_end = EPS_START, EPS_END
eps_curve = [max(eps_end, eps_start - EPS_DECAY * i) for i in range(N_EPISODES)]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(eps_curve, color='#9C27B0', lw=2)
ax.fill_between(range(N_EPISODES), eps_curve, alpha=0.15, color='#9C27B0')
ax.set_title('Decaimento de ε (Exploração → Exploração Gananciosa)', fontsize=13, fontweight='bold')
ax.set_xlabel('Episódio')
ax.set_ylabel('Valor de ε')
ax.axhline(eps_end, linestyle='--', color='gray', lw=1)
ax.text(N_EPISODES * 0.85, eps_end + 0.02, f'ε mínimo = {eps_end}', color='gray')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output_epsilon.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Conclusões

### Resumo dos Resultados

Após o treino de 600 episódios, os três agentes desenvolveram comportamentos distintos 
alinhados com as suas funções de recompensa:

#### 🔵 Agente Seguro
- Tende a realizar **menos operações** de forma geral
- **Realiza lucros rapidamente** para evitar reversões de tendência
- A penalização dupla das perdas leva-o a preferir *não entrar* em momentos de incerteza

#### 🔴 Agente Arrojado  
- Apresenta o **maior número de dias com posição aberta** (HODL)
- Aguarda por movimentos de preço mais pronunciados antes de vender
- Pode apresentar maior retorno mas também maior volatilidade de resultados

#### 🟢 Agente Day Trader
- Realiza **operações muito mais frequentes** que os outros
- A penalização diária de holding força-o a fechar posições rapidamente
- Maior número de trades, menores ganhos por operação

### Reflexão sobre o Q-Learning em Trading

O Q-Learning demonstra-se adequado para este cenário porque:
1. O ambiente é **Markoviano**: a decisão depende apenas do estado atual, não do histórico
2. O espaço de estados é **discreto e limitado** (18 estados), tornando a Q-Table eficiente
3. A **função de recompensa moldável** permite introduzir comportamentos emergentes diferentes

### Limitações e Trabalho Futuro

- **Dados sintéticos**: em produção, usar dados reais do Yahoo Finance (biblioteca `yfinance`)
- **Espaço de estados raso**: adicionar RSI, MACD, Bollinger Bands ao estado aumentaria a precisão
- **Deep Q-Network (DQN)**: para espaços de estados contínuos, substituir a Q-Table por uma rede neural
- **Multi-agente**: simular um mercado com múltiplos agentes a interagir entre si


## Referências

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
- Watkins, C. J. C. H. (1989). *Learning from Delayed Rewards*. PhD Thesis, Cambridge University.
- Material das aulas práticas de Agentes Autónomos, 2025/2026.